# 第四版 FCM：独立硬件验证

由你手动执行，使用新双IP bitstream：FCM + 原V3 pointwise。只依赖板卡已有PYNQ和NumPy，不安装库。
上传前先在电脑解压，将完整 `fcm_v4_test` 文件夹上传为新的同级目录，例如 `/home/xilinx/jupyter_notebooks/SonicArm/fcm_v4_test`。
保留原 `pointwise_test`、`full_model_test`。不要把本包覆盖到前三版。

范围：12个FCM二维卷积、原Q1量化、ReLU、4个残差分支；逐层DDR往返，尚未融合为块内片上流水线。
这不是完整CAM++、精度验收或测速。原QDQ数值差异仍保留；不输出身份通过/拒绝。

**先结束其他使用FPGA的Notebook，逐格执行。若异常，停止并返回完整输出。IP超时/运行中中断后，先断电重启板卡再继续，保持当前kernel直到断电，不要立即释放活动DDR缓冲区。**


In [ ]:
from pathlib import Path
import sys, uuid
sys.dont_write_bytecode = True
ROOT = Path.cwd().resolve()
assert (ROOT/'fcm_v4.bit').is_file() and (ROOT/'fcm_board.py').is_file(), '请从完整 fcm_v4_test 文件夹打开'
sys.path.insert(0, str(ROOT))
from fcm_board import verify_package
from test_fcm_pynq import run_stage
manifest = verify_package()
RUN = ROOT/'test_results'/('run_'+uuid.uuid4().hex)
RUN.mkdir(parents=True)
print('文件校验通过；结果目录：', RUN)
print('本格未加载Overlay；下一格开始使用FPGA。')


## 1. 冒烟与共存回归

本格加载第四版Overlay。验证首卷积、降采样、投影、残差、末卷积以及一组原第三版1×1卷积。等待 `V4_FCM_SMOKE_PASS` 后再运行下一格。


In [ ]:
smoke_report = run_stage('smoke', RUN)


## 2. 完整FCM硬件一致性

48组真实输入均检查INT8/INT32，分别开启/关闭调试写回；随后用真实上层FPGA输出重建两段12层FCM，最后检查2051帧K3卷积的分块接缝。此处使用用于设计验证的300/67帧样例，不替代完整录音评测。


In [ ]:
correctness_report = run_stage('correctness', RUN)


完成后返回两格完整输出以及本次目录中的 `smoke.json`、`correctness.json`。只有这些实际通过后，才接入完整CAM++，比较新增FCM量化漂移与整网时延。保留原第三版评分漂移失败结论。
